In [ ]:
"""
Cell 0 — Setup, Shared Constants, Overall Alpha & Agreement

This cell must be run first. It sets up everything that all subsequent
cells depend on.

Steps:
  1. FILE PATHS       : Define the paths to the three annotators' Excel files.
                        Each file must contain exactly two columns: Text, Label.
  2. LOAD FILES       : Read all three Excel files into DataFrames (df1, df2, df3)
                        and standardise column names to ['Text', 'Label'].
  3. MERGE            : Inner-join all three DataFrames on the 'Text' column,
                        producing a single wide DataFrame ('merged') with columns
                        Text, Tech_A1, Tech_A2, Tech_A3.
  SHARED CONSTANTS    : Define tech_labels, annotators, and pairs — used as-is
                        by Cells 1, 2, and 3 so they are not redefined there.
  4. KRIPPENDORFF'S ALPHA : Compute the three-way nominal Krippendorff's Alpha
                        across all annotators as a single overall reliability score.
  5. PAIRWISE AGREEMENT   : For each of the three annotator pairs (A1-A2, A1-A3,
                        A2-A3), compute the raw count and percentage of texts
                        on which both annotators assigned the same label.
  6. FULL AGREEMENT TABLE : Build a per-text summary showing all three labels and
                        a boolean flag for whether all three annotators agreed.
  7. SAVE OUTPUT      : Write the full agreement table to an Excel file.

Produces:
  merged          — shared DataFrame used by all later cells
  tech_labels     — list of valid technique label strings
  annotators      — list ['A1', 'A2', 'A3']
  pairs           — list of annotator pair tuples
  agreement_df    — per-text agreement table
  agreement_table.xlsx
"""

# ============================================================
# Krippendorff's Alpha + Inter-Annotator Agreement Analysis
# For 3 Excel files containing joke annotations
# Columns: Text, Label
# ============================================================

# Install if needed:
# pip install pandas openpyxl krippendorff

import pandas as pd
import numpy as np
import krippendorff

# ============================================================
# STEP 1: FILE PATHS
# ============================================================

file1 = "./annotation_data/main_study/annotator_1_final.xlsx"
file2 = "./annotation_data/main_study/annotator_2_final.xlsx"
file3 = "./annotation_data/main_study/annotator_3_final.xlsx"

# ============================================================
# STEP 2: LOAD FILES
# ============================================================

df1 = pd.read_excel(file1)
df2 = pd.read_excel(file2)
df3 = pd.read_excel(file3)

# Ensure correct column names
df1.columns = ['Text', 'Label']
df2.columns = ['Text', 'Label']
df3.columns = ['Text', 'Label']

# ============================================================
# STEP 3: MERGE DATA PROPERLY (by Text)
# ============================================================

merged = df1.rename(columns={'Label': 'Tech_A1'})

merged = merged.merge(
    df2.rename(columns={'Label': 'Tech_A2'}),
    on='Text'
)

merged = merged.merge(
    df3.rename(columns={'Label': 'Tech_A3'}),
    on='Text'
)

# ============================================================
# SHARED CONSTANTS
# ============================================================

tech_labels = ['E','A','S','R','M']
annotators  = ['A1', 'A2', 'A3']
pairs       = [('A1','A2'), ('A1','A3'), ('A2','A3')]

# ============================================================
# STEP 4: KRIPPENDORFF'S ALPHA
# ============================================================

print("\n==============================")
print("KRIPPENDORFF'S ALPHA")
print("==============================")

tech_data = [
    merged['Tech_A1'].tolist(),
    merged['Tech_A2'].tolist(),
    merged['Tech_A3'].tolist()
]

alpha_tech = krippendorff.alpha(
    reliability_data=tech_data,
    level_of_measurement='nominal'
)

print(f"Technique Alpha: {round(alpha_tech,4)}")

# ============================================================
# STEP 5: PAIRWISE AGREEMENT
# ============================================================

print("\n==============================")
print("PAIRWISE AGREEMENT")
print("==============================")

for p1, p2 in pairs:
    
    col1 = f'Tech_{p1}'
    col2 = f'Tech_{p2}'
    
    agree = (merged[col1] == merged[col2]).sum()
    total = len(merged)
    
    percent = agree / total * 100
    
    print(f"\nTechnique Agreement {p1}-{p2}: {agree}/{total} = {percent:.2f}%")

# ============================================================
# STEP 6: FULL AGREEMENT TABLE
# ============================================================

print("\n==============================")
print("FULL AGREEMENT TABLE")
print("==============================")

agreement_rows = []

for _, row in merged.iterrows():
    
    tech_vals = [row['Tech_A1'], row['Tech_A2'], row['Tech_A3']]
    tech_agree = len(set(tech_vals)) == 1
    
    agreement_rows.append([
        row['Text'],
        tech_vals,
        tech_agree
    ])

agreement_df = pd.DataFrame(
    agreement_rows,
    columns=[
        'Text',
        'Technique Labels',
        'Full Agreement'
    ]
)
label_info={'E':'ELaboration','A':'Anti-Humor', 'S':'Synergy','R':'Recalibration', 'M':'Miscellaneous' }
print(label_info)
print(agreement_df)

# ============================================================
# STEP 7: SAVE OUTPUT
# ============================================================

output_path = "./data_analysis/agreement_table_for_main_study.xlsx"

agreement_df.to_excel(output_path, index=False)

print(f"\nSaved file at: {output_path}")



KRIPPENDORFF'S ALPHA
Technique Alpha: 0.4605

PAIRWISE AGREEMENT

Technique Agreement A1-A2: 488/746 = 65.42%

Technique Agreement A1-A3: 478/746 = 64.08%

Technique Agreement A2-A3: 566/746 = 75.87%

FULL AGREEMENT TABLE
                                                  Text Technique Labels  \
0    I don't know what the big deal is about Black ...        [E, R, E]   
1    I was playing scattergories with my Palestine ...        [E, S, E]   
2    The only difference between fit and fat is one...        [R, E, S]   
3    Criminals who work in groups should be proud o...        [S, S, S]   
4    I just saw a man repair his monocle with his b...        [E, S, S]   
..                                                 ...              ...   
741  Putting 'not' to the end of your usual joke ma...        [E, R, R]   
742  There are two types of people in the world:___...        [E, R, E]   
743  I love everything french. Some call me a frenc...        [S, S, S]   
744  What do you call octop

In [10]:
"""
Cell 1 — Pairwise Krippendorff's Alpha Between Annotators

Depends on: merged, pairs (defined in Cell 0).

Computes Krippendorff's Alpha separately for each of the three annotator
pairs (A1-A2, A1-A3, A2-A3) rather than the single three-way score from
Cell 0. This isolates which specific pair of annotators agrees most and
which diverges most.

Steps:
  pairwise_alpha()  : Helper function that takes two column names from
                      'merged' and returns their nominal Krippendorff's
                      Alpha rounded to 4 decimal places.
  PAIRS LOOP        : Calls pairwise_alpha() for each annotator pair on
                      the Technique label columns (Tech_A1, Tech_A2, Tech_A3)
                      and prints each result.
  SAVE              : Writes the three pair-wise alpha scores to an Excel file.

Produces:
  result_df         — DataFrame with columns Annotator 1, Annotator 2, Technique Alpha
  pairwise_krippendorff_alpha.xlsx
"""

# ============================================================
# PAIRWISE KRIPPENDORFF'S ALPHA BETWEEN ANNOTATORS
# (Technique ONLY — since new files have Text + Label)
# ============================================================

# merged, pairs already defined in Cell 0

import krippendorff

# ------------------------------------------------------------
# FUNCTION: PAIRWISE KRIPPENDORFF'S ALPHA
# ------------------------------------------------------------

def pairwise_alpha(col1, col2):
    
    data = [
        merged[col1].tolist(),
        merged[col2].tolist()
    ]
    
    alpha = krippendorff.alpha(
        reliability_data=data,
        level_of_measurement='nominal'
    )
    
    return round(alpha, 4)

# ------------------------------------------------------------
# PAIRS
# ------------------------------------------------------------

print("\n====================================")
print("PAIRWISE KRIPPENDORFF'S ALPHA")
print("====================================")

results = []

for a, b in pairs:
    
    tech_alpha = pairwise_alpha(
        f"Tech_{a}",
        f"Tech_{b}"
    )
    
    print(f"\nPair: {a} vs {b}")
    print(f"Technique Alpha = {tech_alpha}")
    
    results.append([a, b, tech_alpha])

# ------------------------------------------------------------
# SAVE RESULTS
# ------------------------------------------------------------

result_df = pd.DataFrame(
    results,
    columns=[
        "Annotator 1",
        "Annotator 2",
        "Technique Alpha"
    ]
)

output_path = "./data_analysis/pairwise_krippendorff_alpha.xlsx"

result_df.to_excel(output_path, index=False)

print(f"\nSaved: {output_path}")



PAIRWISE KRIPPENDORFF'S ALPHA

Pair: A1 vs A2
Technique Alpha = 0.433

Pair: A1 vs A3
Technique Alpha = 0.3975

Pair: A2 vs A3
Technique Alpha = 0.5535

Saved: ./data_analysis/pairwise_krippendorff_alpha.xlsx


In [12]:
"""
Cell 2 — Extended Label-Level Agreement Analysis

Depends on: merged, tech_labels, annotators, pairs (defined in Cell 0).

Breaks down agreement and disagreement at the level of individual labels
and annotator pairs, and reports the label distribution per annotator.

Parts:
  1. OVERALL AGREEMENT / DISAGREEMENT
       For each label (E, A, S, R, M), counts across all texts how many
       times at least 2 of the 3 annotators assigned that label (agreement)
       versus exactly 1 annotator assigned it (disagreement). Highlights
       the label with the highest agreement and the one with the highest
       disagreement.

  2. PAIRWISE LABEL AGREEMENT / DISAGREEMENT
       Repeats the above breakdown for each of the three annotator pairs
       individually, so you can see which pairs agree or disagree on each
       specific label.

  3. LABEL RATIOS PER ANNOTATOR
       For each annotator, shows the percentage share of each label in
       their annotations — useful for spotting individual annotator bias.

  4. AVERAGE RATIOS ACROSS ALL ANNOTATORS
       Pools all three annotators' labels and reports the overall percentage
       share of each label across the entire annotation set.
"""

# ============================================================
# EXTRA ANALYSIS (UPDATED FOR NEW FILES)
# Only Technique Labels (NO Rating)
# ============================================================

# merged, tech_labels, annotators, pairs already defined in Cell 0

import pandas as pd

label_info={'E':'ELaboration','A':'Anti-Humor', 'S':'Synergy','R':'Recalibration', 'M':'Miscellaneous' }
print(label_info)
# ============================================================
# PART 1: OVERALL AGREEMENT / DISAGREEMENT
# ============================================================

print("\n================================================")
print("OVERALL LABEL AGREEMENT / DISAGREEMENT")
print("================================================")

overall_results = []

for label in tech_labels:

    agreement = 0
    disagreement = 0

    for _, row in merged.iterrows():

        vals = [row["Tech_A1"], row["Tech_A2"], row["Tech_A3"]]
        count = vals.count(label)

        if count >= 2:
            agreement += 1
        elif count == 1:
            disagreement += 1

    overall_results.append([label, agreement, disagreement])

tech_result = pd.DataFrame(
    overall_results,
    columns=["Technique", "Agreement", "Disagreement"]
)

print("\nTechnique Labels")
print(tech_result)

print("\nMOST AGREEMENT:")
print(tech_result.loc[tech_result["Agreement"].idxmax()])

print("\nMOST DISAGREEMENT:")
print(tech_result.loc[tech_result["Disagreement"].idxmax()])


# ============================================================
# PART 2: PAIRWISE AGREEMENT / DISAGREEMENT PER LABEL
# ============================================================

print("\n================================================")
print("PAIRWISE LABEL AGREEMENT / DISAGREEMENT")
print("================================================")

for a, b in pairs:

    print(f"\n====================")
    print(f"{a} vs {b}")
    print("====================")

    rows = []

    for label in tech_labels:

        agree = (
            (merged[f"Tech_{a}"] == label) &
            (merged[f"Tech_{b}"] == label)
        )

        agreement = agree.sum()

        disagreement = (
            ((merged[f"Tech_{a}"] == label) &
             (merged[f"Tech_{b}"] != label))
            |
            ((merged[f"Tech_{a}"] != label) &
             (merged[f"Tech_{b}"] == label))
        ).sum()

        rows.append([label, agreement, disagreement])

    temp = pd.DataFrame(
        rows,
        columns=["Technique", "Agreement", "Disagreement"]
    )

    print("\nTechnique")
    print(temp)


# ============================================================
# PART 3: RATIOS OF LABELS FOR EACH ANNOTATOR
# ============================================================

print("\n================================================")
print("LABEL RATIOS FOR EACH ANNOTATOR")
print("================================================")

for ann in annotators:

    print(f"\nTechnique Ratios - {ann}")

    ratio = (
        merged[f"Tech_{ann}"]
        .value_counts(normalize=True)
        * 100
    ).round(2)

    print(ratio)


# ============================================================
# PART 4: AVERAGE RATIOS ACROSS ALL ANNOTATORS
# ============================================================

print("\n================================================")
print("AVERAGE RATIOS")
print("================================================")

all_tech = pd.concat([
    merged["Tech_A1"],
    merged["Tech_A2"],
    merged["Tech_A3"]
])

print("\nAverage Technique Ratio")

print(
    (all_tech.value_counts(normalize=True) * 100)
    .round(2)
)


{'E': 'ELaboration', 'A': 'Anti-Humor', 'S': 'Synergy', 'R': 'Recalibration', 'M': 'Miscellaneous'}

OVERALL LABEL AGREEMENT / DISAGREEMENT

Technique Labels
  Technique  Agreement  Disagreement
0         E        158           159
1         A         10            46
2         S        446            91
3         R         81           122
4         M          1            10

MOST AGREEMENT:
Technique         S
Agreement       446
Disagreement     91
Name: 2, dtype: object

MOST DISAGREEMENT:
Technique         E
Agreement       158
Disagreement    159
Name: 0, dtype: object

PAIRWISE LABEL AGREEMENT / DISAGREEMENT

A1 vs A2

Technique
  Technique  Agreement  Disagreement
0         E        114           188
1         A          6            41
2         S        321           165
3         R         47           111
4         M          0            11

A1 vs A3

Technique
  Technique  Agreement  Disagreement
0         E        101           179
1         A          7            40
2

In [ ]:
"""
Cell 3 — Annotation Count Table and Ratio Table

Depends on: merged, tech_labels, annotators (defined in Cell 0).

Produces two summary tables that make it easy to compare how much each
annotator used each label, both in raw counts and as percentages.

Parts:
  1. TECHNIQUE COUNTS
       Builds a table (rows = labels, columns = A1 / A2 / A3 / Average)
       showing the raw count of each label per annotator, plus the average
       count across the three annotators.

  2. TECHNIQUE RATIOS (%)
       Converts the count table to percentages (each annotator's column
       divided by their total annotation count), then appends an Average %
       column averaged across the three annotators.
"""

# ============================================================
# COUNT OF EACH ANNOTATION + AVERAGE COUNTS
# (Technique ONLY — updated for new files)
# ============================================================

# merged, tech_labels, annotators already defined in Cell 0

import pandas as pd
label_info={'E':'ELaboration','A':'Anti-Humor', 'S':'Synergy','R':'Recalibration', 'M':'Miscellaneous' }
print(label_info)
# ============================================================
# PART 1: TECHNIQUE COUNTS
# ============================================================

print("\n================================================")
print("TECHNIQUE ANNOTATION COUNTS")
print("================================================")

tech_table = pd.DataFrame(index=tech_labels)

for ann in annotators:

    counts = merged[f"Tech_{ann}"].value_counts()

    tech_table[ann] = [
        counts.get(label, 0)
        for label in tech_labels
    ]

# Average count
tech_table["Average"] = tech_table.mean(axis=1).round(2)

print(tech_table)


# ============================================================
# PART 2: TECHNIQUE RATIOS (%)
# ============================================================

print("\n================================================")
print("TECHNIQUE RATIOS (%)")
print("================================================")

tech_ratio = tech_table.copy()

for col in ["A1", "A2", "A3"]:
    total = tech_ratio[col].sum()
    
    if total != 0:
        tech_ratio[col] = (tech_ratio[col] / total * 100).round(2)
    else:
        tech_ratio[col] = 0

# Average %
tech_ratio["Average %"] = (
    tech_ratio[["A1","A2","A3"]].mean(axis=1)
).round(2)

print(tech_ratio)


In [14]:
"""
Cell 4 — Krippendorff's Alpha Per Label via Binary Transformation

Depends on: df1, df2, df3 (loaded in Cell 0).

Computes a per-label Krippendorff's Alpha by binarising annotations: for
each label, every annotation is converted to 1 (annotator used this label)
or 0 (annotator did not). This gives a reliability score specific to each
label rather than a single score across all labels.

Note: this cell creates its own merged DataFrame ('merged_bin') with plain
column names A1, A2, A3 — distinct from the Tech_A1 / Tech_A2 / Tech_A3
columns in 'merged' — so it does not overwrite the shared DataFrame from
Cell 0.

Steps:
  CLEAN LABELS      : Normalise all label strings to uppercase to avoid
                      case-mismatch errors before merging.
  MERGE             : Inner-join the three DataFrames on 'Text' into
                      merged_bin with columns A1, A2, A3.
  to_binary()       : Helper that converts a label column to a binary list
                      for a given target label.
  OVERALL ALPHA     : For each label, computes the three-way nominal
                      Krippendorff's Alpha on the binarised data.
  PAIRWISE ALPHA    : For each label × annotator pair combination, computes
                      the pairwise nominal Krippendorff's Alpha on the
                      binarised data, giving a 15-entry breakdown (5 labels
                      × 3 pairs).

Produces:
  merged_bin        — binarisation-ready merged DataFrame (A1, A2, A3)
  overall_results   — list of [label, alpha] for the three-way scores
  pairwise_results  — list of [label, p1, p2, alpha] for pairwise scores
"""

# ============================================================
# KRIPPENDORFF'S ALPHA PER LABEL (BINARY TRANSFORMATION)
# For 3 annotators (Text, Label)
# ============================================================

# file1/file2/file3 and df1/df2/df3 already loaded in Cell 0
# Note: this cell re-merges with plain A1/A2/A3 column names
#       (distinct from Tech_A1 etc.) for the binary alpha computation

import pandas as pd
import krippendorff
label_info={'E':'ELaboration','A':'Anti-Humor', 'S':'Synergy','R':'Recalibration', 'M':'Miscellaneous' }
print(label_info)
# ------------------------------------------------------------
# CLEAN LABELS (IMPORTANT)
# ------------------------------------------------------------

df1['Label'] = df1['Label'].str.upper()
df2['Label'] = df2['Label'].str.upper()
df3['Label'] = df3['Label'].str.upper()

# ------------------------------------------------------------
# MERGE (by Text)
# ------------------------------------------------------------

merged_bin = df1.rename(columns={'Label': 'A1'})
merged_bin = merged_bin.merge(df2.rename(columns={'Label': 'A2'}), on='Text')
merged_bin = merged_bin.merge(df3.rename(columns={'Label': 'A3'}), on='Text')

# ------------------------------------------------------------
# LABELS
# ------------------------------------------------------------

labels = ['E', 'S', 'R', 'A', 'M']

# ------------------------------------------------------------
# FUNCTION: BINARY TRANSFORMATION
# ------------------------------------------------------------

def to_binary(series, target_label):
    return [1 if x == target_label else 0 for x in series]

# ------------------------------------------------------------
# OVERALL ALPHA PER LABEL
# ------------------------------------------------------------

print("\n======================================")
print("KRIPPENDORFF'S ALPHA PER LABEL (OVERALL)")
print("======================================")

overall_results = []

for label in labels:

    data = [
        to_binary(merged_bin['A1'], label),
        to_binary(merged_bin['A2'], label),
        to_binary(merged_bin['A3'], label)
    ]

    alpha = krippendorff.alpha(
        reliability_data=data,
        level_of_measurement='nominal'
    )

    print(f"{label}: Alpha = {round(alpha,4)}")

    overall_results.append([label, round(alpha,4)])


# ------------------------------------------------------------
# PAIRWISE ALPHA PER LABEL
# ------------------------------------------------------------

print("\n======================================")
print("PAIRWISE ALPHA PER LABEL")
print("======================================")

bin_pairs = [('A1','A2'), ('A1','A3'), ('A2','A3')]

pairwise_results = []

for label in labels:

    print(f"\n--- Label: {label} ---")

    for p1, p2 in bin_pairs:

        data = [
            to_binary(merged_bin[p1], label),
            to_binary(merged_bin[p2], label)
        ]

        alpha = krippendorff.alpha(
            reliability_data=data,
            level_of_measurement='nominal'
        )

        print(f"{p1}-{p2}: {round(alpha,4)}")

        pairwise_results.append([label, p1, p2, round(alpha,4)])


{'E': 'ELaboration', 'A': 'Anti-Humor', 'S': 'Synergy', 'R': 'Recalibration', 'M': 'Miscellaneous'}

KRIPPENDORFF'S ALPHA PER LABEL (OVERALL)
E: Alpha = 0.4075
S: Alpha = 0.5896
R: Alpha = 0.3485
A: Alpha = 0.2585
M: Alpha = 0.0788

PAIRWISE ALPHA PER LABEL

--- Label: E ---
A1-A2: 0.3738
A1-A3: 0.3695
A2-A3: 0.4833

--- Label: S ---
A1-A2: 0.555
A1-A3: 0.5176
A2-A3: 0.695

--- Label: R ---
A1-A2: 0.3727
A1-A3: 0.2562
A2-A3: 0.4126

--- Label: A ---
A1-A2: 0.1985
A1-A3: 0.232
A2-A3: 0.386

--- Label: M ---
A1-A2: -0.0068
A1-A3: 0.1605
A2-A3: 0.0
